# Feature Engineering Of Iris Dataset

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [2]:
df = pd.read_csv('archive (1)/Iris.csv')

In [3]:
df =df.drop(columns=['Id'])

# Handling Missing Values

- Your EDA already shows:

- No missing values are present, resulting in 100% data completeness.

In [4]:
df.isnull().sum()

SepalLengthCm    0
SepalWidthCm     0
PetalLengthCm    0
PetalWidthCm     0
Species          0
dtype: int64

# Handling Outliers

- EDA says: Only a small number of observations appear as outliers, primarily in Sepal Width.

In [5]:
for col in df.select_dtypes(include='number').columns:
    
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    print(f"{col}: {len(outliers)} outliers")

SepalLengthCm: 0 outliers
SepalWidthCm: 4 outliers
PetalLengthCm: 0 outliers
PetalWidthCm: 0 outliers


In [6]:
Q1 = df['SepalWidthCm'].quantile(0.25)
Q3 = df['SepalWidthCm'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

outliers = df[
    (df['SepalWidthCm'] < lower) |
    (df['SepalWidthCm'] > upper)
]

print(outliers)

    SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm          Species
15            5.7           4.4            1.5           0.4      Iris-setosa
32            5.2           4.1            1.5           0.1      Iris-setosa
33            5.5           4.2            1.4           0.2      Iris-setosa
60            5.0           2.0            3.5           1.0  Iris-versicolor


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   SepalLengthCm  150 non-null    float64
 1   SepalWidthCm   150 non-null    float64
 2   PetalLengthCm  150 non-null    float64
 3   PetalWidthCm   150 non-null    float64
 4   Species        150 non-null    object 
dtypes: float64(4), object(1)
memory usage: 6.0+ KB


# Feature Scaling

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = pd.DataFrame(
    scaler.fit_transform(
        df.drop('Species', axis=1)
    ),
    columns=df.drop('Species', axis=1).columns
)

# Feature Encoding

In [9]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['Species'] = le.fit_transform(df['Species'])

# Additional Feature Engineering

In [10]:
# Petal Area
df['PetalArea'] = (
    df['PetalLengthCm'] *
    df['PetalWidthCm']
)

In [11]:
# Sepal Area
df['SepalArea'] = (
    df['SepalLengthCm'] *
    df['SepalWidthCm']
)

In [12]:
# Petal to Sepal Ratio
df['PetalSepalRatio'] = (
    df['PetalArea'] /
    df['SepalArea']
)

# Forward Selection using P-value

In [13]:
import statsmodels.api as sm

def forward_selection(X, y):
    
    selected_features = []
    remaining_features = list(X.columns)

    best_adj_r2 = -float("inf")

    while remaining_features:

        best_candidate = None
        current_best_adj_r2 = -float("inf")

        for feature in remaining_features:

            features_to_fit = selected_features + [feature]

            X_model = sm.add_constant(X[features_to_fit])

            model = sm.OLS(y, X_model).fit()

            if model.rsquared_adj > current_best_adj_r2:
                current_best_adj_r2 = model.rsquared_adj
                best_candidate = feature

        if current_best_adj_r2 > best_adj_r2:

            best_adj_r2 = current_best_adj_r2

            selected_features.append(best_candidate)

            remaining_features.remove(best_candidate)

            print(
                f"Selected {best_candidate} "
                f"(Adjusted R² = {best_adj_r2:.4f})"
            )

        else:
            break

    return selected_features

In [14]:
selected_feature = forward_selection(x,y)

print("Final featue (forward selection):",selected_feature)

NameError: name 'x' is not defined